# DERİ KANSERİ LEZYONLARINI SINIFLANDIRMA

Harun Bürkük - 21627045

Muhammet Subaşi - 21627601

Bu projede deri kanseri lezyon tipini lezyon resimlerine bakarak teşhis koyan bir makine öğrenmesi modeli oluşturmaya çalıştık. 


### İçerik

[Problem](#problem)   

[Veriyi Anlamlandırma](#data_understanding)  

[Veriyi Hazırlama](#data_preparation)

[Veriyi Büyütme](#dataaugmentation) 

[Modelleme](#modeling)  

[Değerlendirme](#evaluation) 
  
[Kaynakça](#references)

### Problem <a class="anchor" id="problem"></a>[](http://)

Dermatologlar için cilt kanserlerinin teşhisinde kullanılan ilk ve en önemli şey ciltte oluşan lezyonların tetkikidir. HAM10000 sayesinde, çeşitli cilt kanserlerine sahip farklı popülasyonlardan farklı kişilerden toplanan bir veri setimiz var. Şimdi dermatologların sinir ağlarını kullanarak lezyonların görüntülerine bakarak kanser türünü nasıl teşhis ettiklerini simüle edeceğiz. Aşağıdaki 7 cilt kanserini teşhis etmeye çalışacağız;

1) İyi huylu keratoz benzeri lezyonlar (solar lentijinler / seboreik keratozlar ve keratozlar gibi liken-planus, bkl)

2) Melanositik nevüsler (nv)

3) Dermatofibroma (df)

4) Melanom (mel)

5) Vasküler lezyonlar (anjiyomlar, anjiyokeratomlar, piyojenik granülomlar ve kanama, vasc)

6) Bazal hücreli karsinom (bcc)

7) Aktinik keratozlar ve intraepitelyal karsinom / Bowen hastalığı (akiec)

In [ ]:
import os
import pandas as pd
import numpy as np
import csv
import matplotlib.pyplot as plt
import seaborn as sns
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import tensorflow 
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report,confusion_matrix
from sklearn import metrics

###  Veriyi Anlama <a class="anchor" id="data_understanding"></a>
Bu veri seti, farklı popülasyonlara ait farklı yaş, cinsiyet ve onay otoritesine ait 10015 dermatoskopik görüntüden oluşmaktadır. 28X28x3 RGB piksel görüntüleri yürütme süresine göre seçiyoruz. Veriler hakkında daha fazla bilgi almak için meta verilere bakalım.


In [ ]:
meta=pd.read_csv('/content/data/HAM10000_metadata.csv')
meta.info()

In [ ]:
tensorflow.test.gpu_device_name()

In [ ]:
meta.head()

In [ ]:
meta.shape

Aşağıda görülebileceği gibi, verilerin çoğu nv (melanositik nevüs) sınıfına aittir. Veri kümesinin geri kalanı aşağı yukarı eşit olarak dağıtılmış görünüyor.

In [ ]:
g = sns.catplot(x="dx", kind="count", palette='rocket', data=meta)
g.fig.set_size_inches(16, 5)

g.ax.set_title('Classes of Skin Cancer', fontsize=18)
g.set_xlabels('Skin Cancer Class', fontsize=12)
g.set_ylabels('Number of Data', fontsize=12)

Aşağıdaki grafiği incelediğimizde, hastalık lezyonlarının çoğunun histopatoloji veya takip muayenesi ile doğrulandığını görüyoruz. Bu, veri setine olan güvenimizi artırır.

In [ ]:
g = sns.catplot(x="dx_type", kind="count", palette='vlag', data=meta)
g.fig.set_size_inches(16, 5)

g.ax.set_title('Confirmation Authority of Skin Cancer', fontsize=18)
g.set_ylabels('Number of Data', fontsize=12)

Deri kanseri türlerinin cinsiyete göre dağılımına baktığımızda genel olarak tüm kanser türlerinin erkeklerde kadınlardan daha yaygın olduğunu söyleyebiliriz.

In [ ]:
g = sns.catplot(x="dx", kind="count", hue="sex", palette='deep', data=meta)
g.fig.set_size_inches(16, 5)

g.ax.set_title('Skin Cancer by Sex', fontsize=18)
g.set_xlabels('Skin Cancer Class', fontsize=12)
g.set_ylabels('Number of Data', fontsize=12)
g._legend.set_title('Sex')

Aşağıdaki grafikte de görebileceğimiz gibi, erkek sayısı kadınlardan fazla ve ayrıca küçük bir bilinmeyen veri var.

In [ ]:
bar, ax = plt.subplots(figsize = (10,10))
plt.pie(meta['sex'].value_counts(), labels = meta['sex'].value_counts().index, autopct="%.1f%%")
plt.title('Gender of Patients', size=16)

Aşağıdaki grafikte veri setinin vücut bölgelerindeki lezyonların konumuna göre dağılımını görüyoruz. Sırt ve alt ekstremite vücut kısımlarında daha fazla lezyon görüldüğünü söyleyebiliriz.

In [ ]:
g = sns.catplot(x="dx", kind="count", hue="localization", palette='bright', data=meta)
g.fig.set_size_inches(16, 9)

g.ax.set_title('Skin Cancer Localization', fontsize=20)
g.set_xlabels('Skin Cancer Class', fontsize=14)
g.set_ylabels('Frequency of Occurance', fontsize=14)
g._legend.set_title('Localization')

Aşağıda hastaların yaşlarının hastalıklara göre dağılımına bakıyoruz.

In [ ]:
g = sns.catplot(x="dx", kind="count", hue="age", palette='bright', data=meta)
g.fig.set_size_inches(16, 9)

g.ax.set_title('Skin Cancer by Age', fontsize=20)
g.set_xlabels('Skin Cancer Class', fontsize=14)
g.set_ylabels('Number of Data', fontsize=14)
g._legend.set_title('Age')

Aşağıda kanser türü ne olursa olsun hastaların yaş dağılımının histogramını görüyoruz.

Bu veri setinde çoğunlukla 40-60 yaş arası hastalardan alınan lezyon örnekleri olduğunu söyleyebiliriz.

In [ ]:
bar, ax = plt.subplots(figsize=(10,10))
sns.histplot(meta['age'])
plt.title('Histogram of Age of Patients', size=16)

# Veriyi Hazırlama <a class="anchor" id="data_preparation"></a>

Veri hazırlama bölümünde öncelikle pikselleri ve hedef etiketlerini (kanser türleri) x ve y diyerek ayırıp y'yi kategorik formata çevirelim.

In [ ]:
df=pd.read_csv('/content/data/hmnist_28_28_RGB.csv')
x=df.drop('label',axis=1)
y=df['label']
x=x.to_numpy()
x=x/255
y=to_categorical(y)

In [ ]:
df['label'].value_counts()

In [ ]:
label={
    'Actinic keratoses':0,
    'Basal cell carcinoma':1,
    'Benign keratosis-like lesions':2,
    'Dermatofibroma':3,
    'Melanocytic nevi':4,
    'Vascular lesions':5,
    'Melanoma':6
}

Verileri 28x28x3 biçiminde yeniden şekillendirdik. 28x28 piksel değerlerini gösterir ve RGB değerleri ise x3'tür.

In [ ]:
x_view=x.reshape(-1,28,28,3)
x = x.reshape(-1,28,28,3)

Veri kümesin bazı örnekleri aşağıda gösterdik.

In [ ]:
plt.figure(figsize=(20,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    img=x_view[i]
    plt.imshow(img)

# Veriyi Büyütme <a class="anchor" id="dataaugmentation"></a>

Veri kümesini yüzde 20'si test kümesi geri kalanı eğitim kümesi olacak şekilde ayırdık.

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=1234)

Veri büyütme bölümünde, modelin performansını ve genelleme yeteneğini geliştirmek için eğitim veri setini genişletmeye çalışıyoruz.

Keras derin öğrenme kütüphanesi tarafından desteklenen ImageDataGenerator sınıfını kullanıyoruz. ImageDataGenerator, orijinal görüntü ile aynı sınıfa ait olan eğitim veri kümesinde görüntülerin dönüştürülmüş sürümlerini oluşturur. Bu dönüşümler, aralık işlemlerini, kaydırmaları, döndürmeleri, yakınlaştırmaları içerir.

In [ ]:
datagen=ImageDataGenerator(rotation_range=20, # rotate the image 20 degrees
                               width_shift_range=0.10, # Shift the pic width by a max of 5%
                               height_shift_range=0.10, # Shift the pic height by a max of 5%
                               rescale=1/255, # Rescale the image by normalzing it.
                               shear_range=0.1, # Shear means cutting away part of the image (max 10%)
                               zoom_range=0.1, # Zoom in by 10% max
                               horizontal_flip=True,
                               vertical_flip=True,
                               fill_mode='nearest')
datagen1=ImageDataGenerator(rotation_range=20, # rotate the image 20 degrees
                               width_shift_range=0.10, # Shift the pic width by a max of 5%
                               height_shift_range=0.10, # Shift the pic height by a max of 5%
                               rescale=1/255, # Rescale the image by normalzing it.
                               shear_range=0.1, # Shear means cutting away part of the image (max 10%)
                               zoom_range=0.1, # Zoom in by 10% max
                               horizontal_flip=True,
                               vertical_flip=True,
                               fill_mode='nearest')

In [ ]:
datagen.fit(X_train)
datagen1.fit(X_test)

# Modelleme <a class="anchor" id="modeling"></a>

Tensorflow.keras.models'deki Sıralı API modellerini kullandık.

İlk katmanda 2 "convolutional" (Conv2D) katman vardır. Bu katman için 32 filtre seçtik. Etkinleştirme fonksiyonu için "relu" kullandık ve tüm görüntüye çekirdek filtresi (3, 3) matrisini uyguladık.

İkinci katmanda "pooling" (MaxPooling2D) katmanı bulunur. Bu katmanın altörnekleme ile aşırı uydurmayı azalttığı söylenebilir. Havuzlama boyutu ne kadar fazlaysa altörnekleme o kadar önemlidir.

"Dropout", ağırlıklarını sıfırlayarak bazı eğitim örneklerini rastgele göz ardı eden bir düzenlilik yöntemidir. Bu teknik, genellemeyi geliştirir ve ayrıca aşırı uyumu azaltır.

"Flatten" katmanı temelde özellikleri 1B vektörüne dönüştüren bir dönüştürme katmanıdır.Bu adımla Cnn, görüntülerden daha fazlasını öğrenmek için "convolutional" ve "pooling" katmanlarını bir araya getirir.

Son olarak, tahmin edilen çok terimli olasılık dağılımlarını elde etmek için çıktılara "softmax" fonksiyonunu kullandık. Eğitimi modellemek için gradyan inişinin "adams" optimizasyonunu kullandık. "Adams" öğrenme oranını ölçeklendirmek için kare gradyanları kullanır.

In [ ]:
from tensorflow.keras.layers import Flatten,Dense,Dropout,BatchNormalization,Conv2D,MaxPooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.metrics import Recall
from tensorflow.keras.optimizers import RMSprop

input_shape = (28, 28, 3)
num_classes = 7

model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3),activation='relu',padding = 'Same',input_shape=input_shape))
model.add(Conv2D(32,kernel_size=(3, 3), activation='relu',padding = 'Same',))
model.add(MaxPooling2D(pool_size = (2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(64, (3, 3), activation='relu',padding = 'Same'))
model.add(Conv2D(64, (3, 3), activation='relu',padding = 'Same'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.40))

model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy',Recall(),tensorflow.keras.metrics.AUC()])
model.summary()

Modelimizin olan CNN'nin hiperparametlerini birden fazla farklı parametreler kullanarak optimize ettik. Aşağıda gördüğünüz gibi kodumuzda batch_size_array, epoch_size_array, learning_rates dizilerimiz var. Birden fazla kere modelimizi sıfırlayarak farklı parametlerle denedik çünkü model eğitimimizin parametre optimizasyonu sırasında kümülatif olmasını istemiyoruz.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau

# early=EarlyStopping(monitor='accuracy',patience=3)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1, mode='min', min_lr=0.0001)

In [ ]:
batch_size_array = [32,64,128]
epoch_size_array = [5,25,50]
learning_rates = [1E-0,1E-2,1E-4]
acc_arr = []
val_loss_arr = []
model.save_weights("model.h5")
for b_size in batch_size_array:
  model.load_weights("model.h5")
  history_model = model.fit(X_train,y_train,batch_size=b_size,epochs=5,validation_split=0.2,callbacks=[reduce_lr])
  acc_arr.append(history_model.history['accuracy'])

max_acc = []
for i in acc_arr:
  print("Max Accuracy:",max(i))
  max_acc.append(max(i))



In [ ]:
a = np.array(batch_size_array)
b = np.array(max_acc)

plt.plot(a, b)
plt.title("Batch Size - Accuracy")
plt.xlabel("Batch Size")
plt.ylabel("Accuracy")
plt.show()

Yukarıda gördüğümüz gibi en iyi doğruluk sonucunu batch_size=128 parametresiyle buluyoruz epoch sayısı ve öğrenme oranı sabit iken. Bundan sonraki eğitimlerde batch_size parametresini 128 olarak kullanacağız.

In [ ]:
acc_arr = []
for e_size in epoch_size_array:
  model.load_weights("model.h5")
  history_model = model.fit(X_train,y_train,batch_size=128,epochs=e_size,validation_split=0.2,callbacks=[reduce_lr])
  acc_arr.append(history_model.history['accuracy'])

for i in acc_arr:
  print("Max Accuracy:",max(i))

In [ ]:
a = np.array(epoch_size_array)
b = np.array(max_acc)

plt.plot(a, b)
plt.title("Epoch Number - Accuracy")
plt.xlabel("Epoch Number")
plt.ylabel("Accuracy")
plt.show()

Yukarıda görüldüğü gibi farklı epoch sayılarıyla denenmiş eğitimlere bakarsak, en iyi doğruluk sonucunun epochs_size=50 ile olduğunu görüyoruz. Bundan sonraki eğitimlerde epoch sayısını 50 olarak kullanacağız.

In [ ]:
acc_arr = []
for lr in learning_rates:
  model.load_weights("model.h5")
  reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1, mode='min', min_lr=lr)
  history_model = model.fit(X_train,y_train,batch_size=128,epochs=50,validation_split=0.2,callbacks=[reduce_lr])
  acc_arr.append(history_model.history['accuracy'])

for i in acc_arr:
  print("Max Accuracy:",max(i))

In [ ]:
a = np.array(learning_rates)
b = np.array(max_acc)

plt.plot(a, b)
plt.title("Learning Rates - Accuracy")
plt.xlabel("Learning Rates")
plt.ylabel("Accuracy")
plt.show()

Yukarıda gördüğünüz gibi en iyi doğruluk sonucunu batch_size ve epoch sayısı en uygunken ve learning_rate=0.001 iken bulduk. Sonuç olarak modelimize en uygun parametleri şöyle bulmuş olduk;

                      batch_size = 128

                      epochs_number = 50

                      learning_rate =  0.0001


Şimdi de modelimizi en uygun parametrelerle son kez eğitelim.

In [ ]:
model.load_weights("model.h5")
early=EarlyStopping(monitor='val_loss',patience=5)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1, mode='min', min_lr=1E-4) 
history_model = model.fit(X_train,y_train,batch_size=128,epochs=50,validation_data=(X_test,y_test),callbacks=[reduce_lr,early])

Hadi şimdi de değerlendirme kısmına geçelim.

# Değerlendirme <a class="anchor" id="evaluation"></a>

Öncelikle modelimizin doğruluk grafiğini göstermek istiyoruz. Görüldüğü gibi modelimizin doğruluk grafigi yükselen bir grafik ortaya koyuyor. 

En uygun parametreler kullanarak 75% gibi bir dogruluk yüzdesi aldık yani diyebiliriz ki modelimiz lezyon fotograflarına bakarak çoğunlukla doğru sonuçlar veriyor.

In [ ]:
plt.figure(figsize=(15,5))
loss=pd.DataFrame(model.history.history)
print(loss.head())
loss=loss[['accuracy','val_accuracy']]
loss.plot(title="Model Accuracy", xlabel="epoch",ylabel="accuracy")


Modelimizin kayıp metrikine baktığımız zaman azalan bir grafiğe sahip olduğunu görüyoruz. Azalan bir grafiğe sahip olması bizim modelimizin başarısını ölçen etmenlerden biridir.

In [ ]:
plt.figure(figsize=(15,5))
loss=pd.DataFrame(model.history.history)
loss=loss[['loss','val_loss']]
loss.plot(title="Model Loss", xlabel="epoch",ylabel="loss")

Ayrıca modelimizin başarısından emin olmak için AUC skoruna bakıyoruz. Bunun için tensorflow.keras.metrics built-in fonksiyonlarını kullanıyoruz.

In [ ]:
plt.figure(figsize=(15,5))
loss=pd.DataFrame(model.history.history)
loss=loss[['auc','val_auc']]
loss.plot(title="AUC Score", xlabel="epoch",ylabel="accuracy")

Aşağıda sınıflandırma raporunun sonuçlarını görebilirsiniz. Tabloya bakarak precisio, recall, f1-skoru sonuçlarını görebilirsiniz.

In [ ]:
predictions=model.predict_classes(X_test)

check=[]
for i in range(len(y_test)):
  for j in range(7):
    if(y_test[i][j]==1):
      check.append(j)
check=np.asarray(check)

print(classification_report(check,predictions))


Aşağıda model.evaluate sonuçlarını test setimiz için görebilirsiniz. AUC skorumuz gördüğünüz gibi oldukça yüksek yani modelimizin oldukça iyi bir iş çıkardığını söyleyebiliriz.

In [ ]:
loss, acc, treshold,treshold2 = model.evaluate(X_test, y_test, verbose=2)


In [ ]:
pred = model.predict(X_test)

matrix = metrics.confusion_matrix(y_test.argmax(axis=1), pred.argmax(axis=1))

ax = plt.axes()    
sns.heatmap(matrix, annot=True, ax=ax,fmt='g')
ax.set_title('Confusion Matrix')
plt.show()

# References <a class="anchor" id="references"></a>

https://machinelearningmastery.com/how-to-configure-image-data-augmentation-when-training-deep-learning-neural-networks/#:~:text=The%20Keras%20deep%20learning%20neural,augmentation%20via%20the%20ImageDataGenerator%20class.&text=Image%20data%20augmentation%20is%20used,of%20the%20model%20to%20generalize.


https://towardsdatascience.com/adam-latest-trends-in-deep-learning-optimization-6be9a291375c


https://keras.io/api/losses/probabilistic_losses/#categoricalcrossentropy-class

https://thesai.org/Downloads/Volume10No6/Paper_38-Hyperparameter_Optimization_in_Convolutional_Neural_Network.pdf
